In [0]:
from pyspark.sql.functions import *

In [0]:
raw_fire_df = spark.read \
    .format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('s3://databricks-external-storage-bucket-eu/fire-department-data/Fire_Department_Calls_For_Service.csv')

In [0]:
display(raw_fire_df)

In [0]:
#It is a good approach to cache the dataframe to improve the performance of consecutive queries on the same dataframe.

raw_fire_df.cache()

#[NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000

In [0]:
renamed_fire_df = raw_fire_df.withColumnRenamed('Call Number', 'CallNumber') \
            .withColumnRenamed('Unit ID', 'UnitID') \
            .withColumnRenamed('Incident Number', 'IncidentNumber') \
            .withColumnRenamed('Call Date', 'CallDate') \
            .withColumnRenamed('Watch Date', 'WatchDate') \
            .withColumnRenamed('Received DtTm', 'ReceivedDtTm') \
            .withColumnRenamed('Entry DtTm', 'EntryDtTm') \
            .withColumnRenamed('Dispatch DtTm', 'DispatchDtTm') \
            .withColumnRenamed('Response DtTm', 'ResponseDtTm') \
            .withColumnRenamed('On Scene DtTm', 'OnSceneDtTm') \
            .withColumnRenamed('Transport DtTm', 'TransportDtTm') \
            .withColumnRenamed('Hospital DtTm', 'HospitalDtTm') \
            .withColumnRenamed('Call Final Disposition', 'CallFinalDisposition') \
            .withColumnRenamed('Available DtTm', 'AvailableDtTm') \
            .withColumnRenamed('Address', 'Address') \
            .withColumnRenamed('City', 'City') \
            .withColumnRenamed('Zipcode of Incident', 'Zipcode') \
            .withColumnRenamed('Battalion', 'Battalion') \
            .withColumnRenamed('Station Area', 'StationArea') \
            .withColumnRenamed('Box', 'Box') \
            .withColumnRenamed('Original Priority', 'OriginalPriority') \
            .withColumnRenamed('Priority', 'Priority') \
            .withColumnRenamed('Final Priority', 'FinalPriority') \
            .withColumnRenamed('ALS Unit', 'ALSUnit') \
            .withColumnRenamed('Call Type Group', 'CallTypeGroup') \
            .withColumnRenamed('Number of Alarms', 'NumberOfAlarms') \
            .withColumnRenamed('Unit Type', 'UnitType') \
            .withColumnRenamed('Unit sequence in call dispatch', 'UnitSequenceInCallDispatch') \
            .withColumnRenamed('Fire Prevention District', 'FirePreventionDistrict') \
            .withColumnRenamed('Supervisor District', 'SupervisorDistrict') \
            .withColumnRenamed('Neighborhooods - Analysis Boundaries', 'Neighborhood') \
            .withColumnRenamed('RowID', 'RowID')

display(renamed_fire_df)

In [0]:
display(raw_fire_df)

In [0]:
renamed_fire_df.printSchema()

In [0]:
fire_df = renamed_fire_df \
    .withColumn('ReceivedDtTm', to_timestamp('ReceivedDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('EntryDtTm', to_timestamp('EntryDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('DispatchDtTm', to_timestamp('DispatchDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('ResponseDtTm', to_timestamp('ResponseDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('OnSceneDtTm', to_timestamp('OnSceneDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('TransportDtTm', to_timestamp('TransportDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('HospitalDtTm', to_timestamp('HospitalDtTm', 'yyyy MMM dd hh:mm:ss a')) \
    .withColumn('AvailableDtTm', to_timestamp('AvailableDtTm', 'yyyy MMM dd hh:mm:ss a'))

In [0]:
display(fire_df)

##### Q1. What San Francisco neighborhoods are in the zip codes 94102 and 94103?

In [0]:
sf_neighborhood_df = fire_df.where("City = 'San Francisco' and Zipcode in (94102, 94103)") \
    .select("Neighborhood", "Zipcode") \
    .distinct()
    
display(sf_neighborhood_df)

##### Q2. How many distinct years of data is in the data set?

In [0]:
# Recommendation is to invoke action always in a new line
# But technically it can be chained with transformations
# It should be the last step
# No other transformation or action can be chained after that because action doesn't return dataframe
years_df = fire_df \
    .select("CallDate") \
    .withColumn("CallDate", year("CallDate")) \
    .distinct()
                     
display(years_df) ## Databricks 'display' which is a utility method behaves kind of similar to Spark show method which is an action

##### Q3. What week of the year in 2018 had the most fire calls?

In [0]:
#DataFrame.count -> action (Spark's most of the transformations return Dataframe object)
#GroupedData.count -> transformation (But Spark groupBy transformation returns GroupedData object)
#Count before group by (on DataFrame) is an action and count after group by (on GroupedData object which is a variant of DF) is a transformation

#In Spark select and where transformation can exchange there position without impacting query performance

most_weekly_call_df = fire_df \
              .select(expr('CallDate as Year'), expr('CallDate as WeekOfTheYear')) \
              .withColumn("Year", year("Year")) \
              .withColumn("WeekOfTheYear", weekofyear("WeekOfTheYear")) \
              .where("Year = 2018" ) \
              .groupBy("WeekOfTheYear") \
              .count() \
              .orderBy(desc("count")) \
              .limit(1) \
              .show() ## -> this is an action